# PhaseGridUNet training and evaluation

Run All trains and evaluates a two-epoch smoke run after the selected dataset has been generated. Smoke training checks integration, not recognition quality. For an existing run, select `RUN_NAME`, run Setup and Configuration, skip Verify dataset and Train model, then run Load saved run onward. Checkpoints must be trusted. Validation supports development; test is held out.


## 1. Setup


In [ ]:
import json
import hashlib
import platform
import random
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
from grid_geometry.dataset.loader import PhaseDataset, inspect_dataset
from grid_geometry.phase.train import Config, dataset_identity, train
from grid_geometry.phase.model import PhaseGridUNet, load_phase_weights
from grid_geometry.phase.evaluate import evaluate, summarize, summarize_global
from grid_geometry.phase.visualize import plot_history, plot_sample
from grid_geometry.training import select_device

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "src" / "grid_geometry").is_dir():
    raise RuntimeError("Open the notebook from the grid-geometry project root or notebooks directory")
DATASETS = PROJECT / "datasets"
RUNS = PROJECT / "runs"
print(f"Project: {PROJECT}")
print(f"Datasets: {DATASETS}")
print(f"Runs: {RUNS}")
print(f"Python {platform.python_version()} | PyTorch {torch.__version__}")

# Identity uses content hashes for the checkpoint and dataset descriptors, plus
# the selected split inventory (size and modification time) for large files.
def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def inventory(paths, root):
    return [[str(p.relative_to(root)), p.stat().st_size, p.stat().st_mtime_ns]
            for p in sorted(paths) if p.is_file()]

def read_completed(destination, expected, required):
    hint = (f"Preserve the existing directory by moving it elsewhere, then rerun: {destination}")
    try:
        metadata = json.loads((destination / "metadata.json").read_text())
        result = json.loads((destination / "report.json").read_text())
        if (metadata.get("settings") != expected or metadata.get("report_sha256") != file_hash(destination / "report.json")
                or not isinstance(result, dict) or not all(k in result for k in required)):
            raise ValueError("metadata or report does not match")
        return result
    except (OSError, ValueError, TypeError) as error:
        raise ValueError(f"Incomplete, legacy, or incompatible evaluation at {destination}. {hint}") from error


## 2. Configuration

Choose a fresh run name for training. `auto` selects the trainer's available device; requested AMP is effective only on CUDA. The saved run configuration and dataset identity control subsequent inspection.


In [ ]:
PRESET = "smoke"
DATASET_NAME = PRESET
RUN_NAME = PRESET
DEVICE = "auto"

presets = {"smoke": {"epochs": 2, "batch_size": 8},
           "development": {"epochs": 50, "batch_size": 16}}
if PRESET not in presets:
    raise ValueError(f"Unknown preset: {PRESET}")
for label, name in (("dataset", DATASET_NAME), ("run", RUN_NAME)):
    if not name or name in {".", ".."} or Path(name).name != name:
        raise ValueError(f"Use a single {label} directory name: {name!r}")
dataset_root = (DATASETS / DATASET_NAME).resolve()
run_dir = (RUNS / RUN_NAME).resolve()
training = dict(workers=0, optimizer="adamw", learning_rate=0.001,
                weight_decay=0.0001, scheduler="none", seed=42, device=DEVICE,
                amp=True, occupancy_weight=1.0, selection_metric="loss", patience=10,
                **presets[PRESET])
config = Config(dataset_root=str(dataset_root), output_dir=str(run_dir), **training)
print(f"Preset: {PRESET}; model: PhaseGridUNet; input: 384 x 384; epochs: {training['epochs']}; batch: {training['batch_size']}")
print(f"Requested device: {DEVICE}; requested AMP: {training['amp']}")


## 3. Verify dataset

The production inspection streams samples once to check pairing, manifest coverage, board isolation, geometry targets and masks. The saved generator settings determine split counts.


In [ ]:
for name in ("config.json", "manifest.jsonl", "statistics.json"):
    if not (dataset_root / name).is_file():
        raise FileNotFoundError(dataset_root / name)
dataset_config = json.loads((dataset_root / "config.json").read_text())
for key, expected in (("generator", "phase-grid-v1"), ("generator_revision", 1), ("status", "complete")):
    if dataset_config.get(key) != expected:
        raise ValueError(f"Dataset {key}: expected {expected!r}, got {dataset_config.get(key)!r}")
if tuple(dataset_config["input_size"]) != (384, 384):
    raise ValueError("This workflow requires the model-compatible 384 x 384 input size")
verification = inspect_dataset(dataset_root)
if any(verification["split_counts"][split] == 0 for split in ("train", "val", "test")):
    raise ValueError("The complete workflow requires nonempty train, val and test splits")
print(f"Verified dataset: {dataset_root}")
print("Split counts: " + ", ".join(f"{k}={v}" for k, v in verification["split_counts"].items()))


## 4. Train model

Fresh runs refuse any existing directory. To resume separately, reconstruct `Config` from the saved `last.pt` configuration, change only the total epoch limit and, if needed, device/output directory, then call `train(resume_config, resume=True)`. The trainer checks dataset identity and compatible settings and restores the full training state.


In [ ]:
if run_dir.exists():
    raise FileExistsError(f"Choose a fresh run name: {run_dir}")
print(f"Training destination: {run_dir}")
history = train(config)


## 5. Load saved run

`last.pt` contains complete history; `best.pt` supplies evaluation weights. The saved dataset path must still be accessible and its identity must match. The identity covers the configuration, manifest and train/validation inventory, but does not hash test image contents. Older checkpoints may lack summary and performance metadata; unavailable values are shown explicitly.


In [ ]:
inspect_dir = (RUNS / RUN_NAME).resolve()
last_path = inspect_dir / "last.pt"
best_path = inspect_dir / "best.pt"
for path in (last_path, best_path):
    if not path.is_file():
        raise FileNotFoundError(path)
state = torch.load(last_path, map_location="cpu", weights_only=False)
saved_config = state.get("config")
if not isinstance(saved_config, dict) or not saved_config.get("dataset_root"):
    raise ValueError("Saved checkpoint lacks dataset configuration")
if not isinstance(state.get("dataset_identity"), str) or not state["dataset_identity"]:
    raise ValueError("Saved checkpoint lacks dataset identity")
saved_root = Path(saved_config["dataset_root"]).resolve()
if dataset_identity(saved_root) != state["dataset_identity"]:
    raise ValueError("Saved dataset identity differs from the current dataset inventory")
best_state = torch.load(best_path, map_location="cpu", weights_only=False)
if best_state.get("dataset_identity") != state["dataset_identity"]:
    raise ValueError("Best and last checkpoints have different dataset identities")
history = state["history"]
if not history:
    raise ValueError("Saved checkpoint has no training history")
summary_path = inspect_dir / "summary.json"
summary = json.loads(summary_path.read_text()) if summary_path.is_file() else {}
print(f"Run: {inspect_dir}")
print(f"Dataset: {saved_root}")
print(f"Last checkpoint: {last_path}")
print(f"Best checkpoint: {best_path}")
print(f"Resolved training: epochs={saved_config.get('epochs')}, batch={saved_config.get('batch_size')}, optimizer={saved_config.get('optimizer')}, learning rate={saved_config.get('learning_rate')}")
def available(value):
    return "unavailable" if value is None else value
outcomes = {
    "completed_epochs": summary.get("completed_epochs", state.get("epoch")),
    "best_epoch": summary.get("best_epoch", state.get("best_epoch")),
    "best_value": summary.get("best_value", state.get("best")),
    "stopping_reason": summary.get("stopping_reason", state.get("stopping_reason")),
    "device_requested": summary.get("device_requested", saved_config.get("device")),
    "device_effective": summary.get("device_effective", state.get("environment", {}).get("device")),
    "amp_requested": summary.get("amp_requested", saved_config.get("amp")),
    "amp_effective": summary.get("amp_effective", state.get("amp_effective")),
    "total_training_seconds": summary.get("total_training_seconds"),
    "aggregate_samples_per_second": summary.get("aggregate_samples_per_second"),
    "successful_updates": summary.get("successful_updates"),
    "skipped_updates": summary.get("skipped_updates"),
}
for key, value in outcomes.items():
    if key == "aggregate_samples_per_second":
        rates = value if isinstance(value, dict) else {}
        formatted = ", ".join(f"{split}={rates[split]:.2f}" if rates.get(split) is not None else f"{split}=unavailable" for split in ("train", "validation"))
        print(f"Aggregate throughput: {formatted} samples/s")
    else:
        text = f"{value:.4f}" if isinstance(value, float) and key == "best_value" else f"{value:.2f}" if isinstance(value, float) else available(value)
        print(f"{key.replace('_', ' ').capitalize()}: {text}")


## 6. Analyze training history

Epoch target metrics aggregate component sums over valid pixels across each split; they are not averages of individual batch optimization objectives. Timing excludes checkpoint writes, and accelerator timing is not synchronized.


In [ ]:
fig = plot_history(history)
best_epoch = outcomes["best_epoch"]
if best_epoch is not None:
    fig.axes[0].axvline(best_epoch, color="black", linestyle=":", label=f"best epoch {best_epoch}")
    fig.axes[0].legend()
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for split in ("train", "validation"):
    for metric in ("phase_loss", "occupancy_loss", "phase_mae_radians_x", "phase_mae_radians_y", "iou"):
        points = [(row["epoch"], row[split]["metrics"]["all"].get(metric)) for row in history]
        points = [(x, y) for x, y in points if y is not None]
        if points:
            ax = axes[0] if metric.endswith("loss") else axes[2] if metric == "iou" else axes[1]
            ax.plot(*zip(*points), marker="o", markersize=3, label=f"{split} {metric}")
for ax, title in zip(axes, ("Loss components", "Phase angular error (radians)", "Occupancy IoU")):
    ax.set(title=title, xlabel="Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    if ax.lines: ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, title, series in (
    (axes[0], "Learning rate", [("learning rate", lambda r: r.get("learning_rate"))]),
    (axes[1], "Epoch duration (s)", [("epoch", lambda r: r.get("performance", {}).get("epoch_seconds"))]),
    (axes[2], "Throughput (samples/s)", [(s, lambda r, s=s: r[s].get("samples_per_second")) for s in ("train", "validation")]),
):
    for label, get in series:
        points = [(r["epoch"], get(r)) for r in history if get(r) is not None]
        if points: ax.plot(*zip(*points), marker="o", markersize=3, label=label)
    ax.set(title=title, xlabel="Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    if ax.lines: ax.legend(fontsize=8)
    else: ax.text(.5, .5, "Unavailable", ha="center", transform=ax.transAxes)
fig.tight_layout()
plt.show()


## 7. Evaluate model
Complete compatible reports are reused. Legacy, incomplete or incompatible directories are protected: move them to a safe location outside `evaluation/val` or `evaluation/test` before evaluating again. The seeded limit is for debugging. Complete-board dimensions and boundaries are scored only where observable; inference and decoding timing is unavailable.


In [ ]:
EVALUATION_LIMIT = None  # positive integer for a seeded debugging subset
EVALUATION_SEED = 42
if EVALUATION_LIMIT is not None and EVALUATION_LIMIT <= 0:
    raise ValueError("EVALUATION_LIMIT must be positive or None")
evaluation_root = inspect_dir / "evaluation"
reports = {}
for split in ("val", "test"):
    destination = evaluation_root / split
    expected = {"schema": 1, "checkpoint_sha256": file_hash(best_path),
                "dataset_root": str(saved_root), "dataset_identity": dataset_identity(saved_root),
                "inventory": inventory([p for p in (saved_root / split).glob("*") if p.suffix in (".png", ".json")], saved_root),
                "split": split, "limit": EVALUATION_LIMIT, "seed": EVALUATION_SEED,
                "device": str(select_device(DEVICE)), "global_geometry": True,
                "evaluator_sha256": file_hash(PROJECT / "src/grid_geometry/phase/evaluate.py")}
    if destination.exists():
        report = read_completed(destination, expected, ("records", "sample_ids", "target_metrics", "geometry", "global_geometry_single_crop"))
        if report.get("split") != split or len(report["records"]) != len(report["sample_ids"]) or any("global" not in r for r in report["records"]):
            raise ValueError(f"Incomplete evaluation report at {destination}; preserve it by moving the directory elsewhere before rerunning")
        print(f"Loaded evaluation: {destination}")
    else:
        destination.mkdir(parents=True, exist_ok=False)
        print(f"Evaluation destination: {destination}")
        report = evaluate(best_path, saved_root, split=split, device=DEVICE,
                          limit=EVALUATION_LIMIT, seed=EVALUATION_SEED,
                          output=destination / "report.json", global_geometry=True)
        (destination / "metadata.json").write_text(json.dumps({"settings": expected, "report_sha256": file_hash(destination / "report.json")}, indent=2) + "\n")
    reports[split] = report
    total = len(PhaseDataset(saved_root, split, require_nonempty=True))
    print(f"{split}: evaluated {len(report['records'])} of {total} samples; subset seed {EVALUATION_SEED}")
    def display_values(title, values):
        print(title)
        for key, value in values.items():
            if isinstance(value, dict):
                display_values("  " + key.replace("_", " ") + ":", value)
            elif isinstance(value, (int, float)) and not isinstance(value, bool):
                precision = 4 if "loss" in key else 2
                print(f"  {key.replace('_', ' ')}: {value:.{precision}f}" if isinstance(value, float) else f"  {key.replace('_', ' ')}: {value}")
            elif value is None:
                print(f"  {key.replace('_', ' ')}: unavailable")
    print("Network target metrics (pixel weighted):")
    print(f"  {'group':12} {'samples':>8} {'phase pixels':>13} {'occ pixels':>12} {'loss':>9} {'phase loss':>11} {'occ loss':>9} {'angle x/y (rad)':>19} {'IoU':>7}")
    def fmt(value, digits=2):
        return "unavailable" if value is None else f"{value:.{digits}f}"
    for group, m in report["target_metrics"].items():
        angles = f"{fmt(m['phase_mae_radians_x'])}/{fmt(m['phase_mae_radians_y'])}"
        print(f"  {group:12} {m['samples']:8d} {m['phase_pixels']:13d} {m['occupancy_pixels']:12d} {fmt(m['loss'], 4):>9} {fmt(m['phase_loss'], 4):>11} {fmt(m['occupancy_loss'], 4):>9} {angles:>19} {fmt(m['iou']):>7}")
    local = report["geometry"]
    display_values("Historical local decoder and validator:", {k: v for k, v in local.items() if k not in ("by_category", "decoder_reasons", "validation_reasons")})
    display_values("Global single-crop recognition (complete and partial reported separately):", report["global_geometry_single_crop"])
    print("Inference and decoding timing: unavailable")


In [ ]:
# Join evaluated records to saved sample metadata; no inference or decoding is repeated.
def sample_metadata(split, sample_id):
    return json.loads((saved_root / split / f"{sample_id}.json").read_text())

def subgroup_labels(meta):
    labels = [f"crop: {meta.get('category', 'unknown')}"]
    if meta.get("geometry"):
        labels.append(f"pitch x bin: {meta.get('actual_pitch_bin_x') or 'unknown'}")
        transform = meta.get("transform") or {}
        scales = [transform.get("scale_x"), transform.get("scale_y")]
        if all(isinstance(v, (int, float)) for v in scales):
            if max(scales) > 1: labels.append("upsampled (any axis)")
            if min(scales) < 1: labels.append("downsampled (any axis)")
    else:
        labels.append(f"negative category: {meta.get('category', 'unknown')}")
    return labels

metadata_by_split = {}
for split, report in reports.items():
    metadata = {r["sample_id"]: sample_metadata(split, r["sample_id"]) for r in report["records"]}
    metadata_by_split[split] = metadata
    groups = {}
    for record in report["records"]:
        for label in subgroup_labels(metadata[record["sample_id"]]):
            groups.setdefault(label, []).append(record)
    print(f"{split} subgroup geometry (counts and rates use each group's available denominators):")
    print(f"  {'Group':32} {'n':>5} {'+/−':>9} {'local recovery':>14} {'negative FA':>12} {'complete correct/accepted/total':>31} {'negative accepted/total':>24}")
    for label, records in sorted(groups.items()):
        local = summarize(records)
        global_result = summarize_global(records)
        recovery = local["correct_recovery_positive"]
        false_accept = local["false_acceptance_negative_rate"]
        recovery = "unavailable" if recovery is None else f"{recovery:.2f}"
        false_accept = "unavailable" if false_accept is None else f"{false_accept:.2f}"
        complete = f"{global_result['complete_correct_accepted']}/{global_result['complete_accepted']}/{global_result['complete_samples']}"
        negative = f"{global_result['negative_accepted']}/{global_result['negative_samples']}"
        counts = f"{local['positives']}/{local['negatives']}"
        print(f"  {label:32} {len(records):5d} {counts:>9} {recovery:>14} {false_accept:>12} {complete:>31} {negative:>24}")


## 8. Visualize predictions

Selection is bounded and deduplicated, using the evaluated validation records and their saved metadata. Failure selection is restricted to that evaluated set (which may be a debugging subset). The production plot shows ground-truth and predicted phase/occupancy fields, masks and decoded local geometry.


In [ ]:
DIAGNOSTIC_SPLIT = "val"
records = reports[DIAGNOSTIC_SPLIT]["records"]
metadata = metadata_by_split[DIAGNOSTIC_SPLIT]
rng = random.Random(42)
shuffled = rng.sample(records, len(records))
selected = {}
def select(record, reason):
    if record is not None:
        selected.setdefault(record["sample_id"], []).append(reason)
def first(predicate):
    return next((r for r in shuffled if predicate(r, metadata[r["sample_id"]])), None)
for category in ("complete", "interior", "side", "corner", "negative", "hard_negative"):
    select(first(lambda r, m, category=category: m.get("category") == category), category)
positive = [r for r in records if metadata[r["sample_id"]].get("geometry")]
if positive:
    pitch = lambda r: metadata[r["sample_id"]]["geometry"]["pitch_x"]
    select(min(positive, key=pitch), "smallest horizontal pitch")
    select(max(positive, key=pitch), "largest horizontal pitch")
for label, predicate in (
    ("upsampled", lambda m: max((m.get("transform") or {}).get("scale_x", 1), (m.get("transform") or {}).get("scale_y", 1)) > 1),
    ("downsampled", lambda m: min((m.get("transform") or {}).get("scale_x", 1), (m.get("transform") or {}).get("scale_y", 1)) < 1),
    ("padded", lambda m: m.get("padding_fraction", 0) > 0),
):
    select(first(lambda r, m, predicate=predicate: predicate(m)), label)
select(first(lambda r, m: r["positive"] and (not r["local_accepted"] or r["local_correct"] is False)), "local geometry failure")
select(first(lambda r, m: not r["positive"] and (r["local_accepted"] or r["global"]["accepted"])), "false acceptance")
print(f"Visualization split: {DIAGNOSTIC_SPLIT}; failure selection: {len(records)} evaluated samples, limit={EVALUATION_LIMIT}, seed={EVALUATION_SEED}")
print(f"Selected {len(selected)} distinct samples")
dataset = PhaseDataset(saved_root, DIAGNOSTIC_SPLIT, require_nonempty=True)
indices = {name: i for i, name in enumerate(dataset.names)}
device = select_device(DEVICE)
model = PhaseGridUNet()
load_phase_weights(model, best_state)
model.to(device).eval()
for sample_id, reasons in selected.items():
    print(f"{sample_id}: {', '.join(reasons)}")
    plot_sample(model, dataset.load_sample(indices[sample_id]), device=device)
    plt.show()
